# wrangle / household / networth

**Intent**
- Gives constants, with the starting wealth the portfolio models need, and must:
  - Give `household_networth_current_dollars`, net worth at the latest month end.
  - Give `household_networth_savings_rate_pct`, the savings rate over the period.
  - Close the loop between the plan's forecasts and what happens.

**Data**
- Personal data is git-ignored in `data/` and `project/`, with synthetic placeholders committed.
- `training` is twelve synthetic months of net worth, income and spending.
- It is synthetic until the `bank` table exists.
- The `Balances` tab of the `Bank` workbook holds a reading per account over time from 2021-05.

**Status**
- The real series is not loaded and nothing is released.
- It carries no evaluation, so the golden replay and the Train section assertions are the gate.

**Future**
- Overall: the work still to do before the first release:
  - Add a `bank` table to `wrangle`.
  - Build the net worth series, reconcile it to income less spending, and separate saving from market return.
  - Add the tracking fan against the base, low and high paths once `portfolio/return` is real.
  - Add property, super and other assets with dated values, and liabilities with their rates.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("wrangle/household/networth")

ws.inputs[0] = ws.inputs[0]

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from mlmodels import collect_run

collect_run(
    ws,
    1000,
    label="Twelve synthetic months of net worth, income and spending",
    name="household_networth_training",
    source="builder",
    build=lambda: {
        "month": list(range(12)),
        "net_worth_dollars": [1000000.0 + 5000.0 * month for month in range(12)],
        "income_dollars": [12000.0] * 12,
        "spending_dollars": [9000.0] * 12,
    },
)

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "household_networth_training"))
print("training", ws.training.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
income = float(ws.training["income_dollars"].sum())
spending = float(ws.training["spending_dollars"].sum())
ws.params = dict(zip(ws.outputs, [round(float(ws.training["net_worth_dollars"][-1]), 0), round((income - spending) / income * 100.0, 2)], strict=True))
print("constants", ws.params)
assert list(ws.params.values()) == [1055000.0, 25.0], "the summary did not give the latest net worth and the savings rate"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

define_model(ws, model=1000, flavour="static", source=ws.params, label="Latest net worth and the savings rate")

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = {ws.inputs[0]: [1.0, None], "expect": ["ok", "error"]}
package_model(ws, 1000, golden, target={"kind": "local"})